In [1]:
import pandas as pd
import datasets
import numpy as np
import os
import transformers
import torch

In [2]:
'''from transformers import pipeline
classifier = pipeline("zero-shot-classification",model="Azma-AI/deberta-base-multi-label-classifier")

text = "one day I will see the world"
candidate_labels = ['travel', 'cooking', 'dancing']
classifier(text, candidate_labels)'''

'from transformers import pipeline\nclassifier = pipeline("zero-shot-classification",model="Azma-AI/deberta-base-multi-label-classifier")\n\ntext = "one day I will see the world"\ncandidate_labels = [\'travel\', \'cooking\', \'dancing\']\nclassifier(text, candidate_labels)'

In [3]:
#test_labels = pd.read_csv(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english\labels.tsv", encoding="utf-8", sep="\t", header=0)
#test_sentences=pd.read_csv(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english\sentences.tsv", encoding="utf-8", sep="\t", header=0)
train_labels= pd.read_csv(r".\Dataset Touché\valueeval24\training-english\4_labels.tsv", encoding="utf-8", sep="\t", header=0)
train_sentences= pd.read_csv(r".\Dataset Touché\valueeval24\training-english\sentences.tsv", encoding="utf-8", sep="\t", header=0)



In [4]:
labels_frame = pd.merge(train_labels, train_sentences, on=["Text-ID", "Sentence-ID"], how="inner")
labels_frame
labels_matrix = np.zeros((labels_frame.shape[0], 5))
labels_matrix

array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       ...,
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

In [5]:
train_labels.info()
#train_sentences.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44758 entries, 0 to 44757
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Text-ID             44758 non-null  object
 1   Sentence-ID         44758 non-null  int64 
 2   Openness to change  44758 non-null  int64 
 3   Self-enhancement    44758 non-null  int64 
 4   Conservation        44758 non-null  int64 
 5   Self-transcendence  44758 non-null  int64 
 6   No Value            44758 non-null  int64 
dtypes: int64(6), object(1)
memory usage: 2.4+ MB


In [6]:
#dataset = 'values_labels'
# dataset = 'subclasess_labels'
labels = ["Openness to change", "Self-enhancement", "Conservation", "Self-transcendence", "No Value"]
#labels = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels = len(labels)

In [7]:
from transformers import DebertaV2Tokenizer, DebertaForSequenceClassification, Trainer, TrainingArguments

# Charger le tokenizer et le modèle DeBERTa
model_name = "Azma-AI/deberta-base-multi-label-classifier"


#"microsoft/deberta-base"
tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)



In [8]:
 '''#Create torch dataset
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None and len(self.labels) > 0:  # Vérification pour éviter les erreurs de condition sur un tableau
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])'''


'#Create torch dataset\nclass Dataset(torch.utils.data.Dataset):\n   def __init__(self, encodings, labels=None):\n       self.encodings = encodings\n       self.labels = labels\n\n   def __getitem__(self, idx):\n       item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}\n       if self.labels is not None and len(self.labels) > 0:  # Vérification pour éviter les erreurs de condition sur un tableau\n           item["labels"] = torch.tensor(self.labels[idx])\n       return item\n\n   def __len__(self):\n       return len(self.encodings["input_ids"])'

In [9]:
'''def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, "4_labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    train_tokenized = tokenizer(data_frame["Text"].iloc[:10000].to_list(),padding=True,truncation=True,max_length=512)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
        labels_matrix = np.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        #encoded_sentences["labels"] = labels_matrix.tolist()
    

    encoded_sentences = Dataset(train_tokenized,labels_matrix[10000:])
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()'''

'def load_dataset(directory, tokenizer, load_labels=True):\n    sentences_file_path = os.path.join(directory, "sentences.tsv")\n    labels_file_path = os.path.join(directory, "4_labels.tsv")\n    \n    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)\n    train_tokenized = tokenizer(data_frame["Text"].iloc[:10000].to_list(),padding=True,truncation=True,max_length=512)\n\n    if load_labels and os.path.isfile(labels_file_path):\n        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)\n        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")\n        labels_matrix = np.zeros((labels_frame.shape[0], len(labels)))\n        for idx, label in enumerate(labels):\n            if label in labels_frame.columns:\n                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)\n        #encoded_sentences["labels"] = labels_matrix.tolist()\n    \n\n    encoded_sentence

In [10]:
def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, f"4_labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    encoded_sentences = tokenizer(data_frame["Text"].iloc[:1000].to_list(),padding=True,truncation=True,max_length=512)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
        labels_matrix = np.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        labels_matrix=labels_matrix[:1000]
        encoded_sentences["labels"] = labels_matrix.tolist()

    encoded_sentences = datasets.Dataset.from_dict(encoded_sentences)
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()

In [11]:
#directory_test=r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english"
directory_train=r".\Dataset Touché\valueeval24\training-english"
directory_validation=r".\Dataset Touché\valueeval24\validation-english"

#encoded_sentences_test, text_ids_test, sentence_ids_test = load_dataset(directory_test, tokenizer)
encoded_sentences_train,text_ids_train, sentence_ids_train = load_dataset(directory_train, tokenizer)
encoded_sentences_validation,text_ids_validation, sentence_ids_validation = load_dataset(directory_validation, tokenizer)

In [12]:
encoded_sentences_train[3]

{'input_ids': [1,
  287,
  58521,
  11616,
  320,
  20473,
  6646,
  285,
  1663,
  264,
  266,
  76250,
  689,
  6536,
  1378,
  267,
  1217,
  261,
  364,
  1760,
  864,
  265,
  14145,
  2932,
  2754,
  2664,
  265,
  1083,
  3165,
  23495,
  276,
  268,
  688,
  876,
  260,
  2,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

In [13]:
'''# Use a Subset of the Dataset
def create_subset(dataset, fraction=0.1):
    subset_size = int(len(dataset) * fraction)
    return dataset.select(range(subset_size))

subset_fraction = 0.1
encoded_sentences_train_subset = create_subset(encoded_sentences_train, subset_fraction)
encoded_sentences_test_subset = create_subset(encoded_sentences_validation, subset_fraction)'''

'# Use a Subset of the Dataset\ndef create_subset(dataset, fraction=0.1):\n    subset_size = int(len(dataset) * fraction)\n    return dataset.select(range(subset_size))\n\nsubset_fraction = 0.1\nencoded_sentences_train_subset = create_subset(encoded_sentences_train, subset_fraction)\nencoded_sentences_test_subset = create_subset(encoded_sentences_validation, subset_fraction)'

In [14]:
from transformers import DebertaV2ForSequenceClassification, AutoTokenizer

num_labels = len(labels)

# Charger le modèle
model = DebertaV2ForSequenceClassification.from_pretrained(model_name, num_labels=num_labels,ignore_mismatched_sizes=True
)  # 3 catégories : travel, cooking, dancing

# Configurer le modèle pour la classification multi-label
model.config.problem_type = "multi_label_classification"


'''
model = DebertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) 
for param in model.parameters():
    param.requires_grad=False  '''

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at Azma-AI/deberta-base-multi-label-classifier and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([5]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([5, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


'\nmodel = DebertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) \nfor param in model.parameters():\n    param.requires_grad=False  '

In [15]:
'''import torch.nn as nn
model.classifier = nn.Sequential(
    nn.ReLU(),
    #nn.Dropout(0.3),  
    nn.Linear(model.config.hidden_size, len(labels)), 
    nn.Sigmoid()
)''' 

'import torch.nn as nn\nmodel.classifier = nn.Sequential(\n    nn.ReLU(),\n    #nn.Dropout(0.3),  \n    nn.Linear(model.config.hidden_size, len(labels)), \n    nn.Sigmoid()\n)'

In [16]:
'''import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(p):
    # Unpack predictions and labels
    preds, labels = p
    # Apply a threshold to predict label presence
    preds = (preds >= 0.5).astype(int)  # Adjust the threshold as needed

    # Calculate multi-label metrics
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='samples', zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }'''

'import numpy as np\nfrom sklearn.metrics import precision_recall_fscore_support, accuracy_score\n\ndef compute_metrics(p):\n    # Unpack predictions and labels\n    preds, labels = p\n    # Apply a threshold to predict label presence\n    preds = (preds >= 0.5).astype(int)  # Adjust the threshold as needed\n\n    # Calculate multi-label metrics\n    accuracy = accuracy_score(labels, preds)\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        labels, preds, average=\'samples\', zero_division=0\n    )\n\n    return {\n        "accuracy": accuracy,\n        "precision": precision,\n        "recall": recall,\n        "f1": f1\n    }'

In [17]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(pred):
    logits, labels = pred
    probs = torch.sigmoid(torch.tensor(logits))
    
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    best_threshold, best_f1 = 0, 0
    for t in thresholds:
        prediction = (probs >= t).int()
        _, _, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples')
        if f1 > best_f1:
            best_threshold, best_f1 = t, f1
    # best_threshold = 0.5
    
    prediction = (probs >= best_threshold).int()
    max_probs = probs.argmax(dim=1)
    for i in range(prediction.shape[0]):
        if prediction[i].sum() == 0:  # No label selected
            prediction[i, max_probs[i]] = 1  # Assign the label with max probability

    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, prediction)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [18]:

#from transformers import DataCollatorWithPadding

 #Créer un collateur de données qui va appliquer le padding aux séquences
#data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [19]:
model.train()

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): StableDropout()
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): StableDropout()
              (dropout): StableDropout()
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine

In [20]:
from transformers import TrainingArguments


args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,             
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    warmup_ratio=0.2
)

optimizer = torch.optim.AdamW(
    model.parameters(),  # Parameters to optimize
    lr=2e-5,             # Learning rate
    weight_decay=0.01    # Weight decay for regularization
)



trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_sentences_train,
    eval_dataset=encoded_sentences_validation,
    #data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
    )

c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [21]:
trainer.train()
#trainer.train(resume_from_checkpoint=True)


  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.3241300582885742, 'eval_accuracy': 0.702, 'eval_f1': 0.702, 'eval_precision': 0.702, 'eval_recall': 0.702, 'eval_runtime': 166.8122, 'eval_samples_per_second': 5.995, 'eval_steps_per_second': 0.749, 'epoch': 1.0}


  0%|          | 0/125 [00:00<?, ?it/s]

c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

...

{'eval_loss': 0.31595295667648315, 'eval_accuracy': 0.502, 'eval_f1': 0.6553, 'eval_precision': 0.61575, 'eval_recall': 0.748, 'eval_runtime': 166.0095, 'eval_samples_per_second': 6.024, 'eval_steps_per_second': 0.753, 'epoch': 3.0}
{'loss': 0.3118, 'grad_norm': 0.7509270310401917, 'learning_rate': 0.0, 'epoch': 4.0}


  0%|          | 0/125 [00:00<?, ?it/s]

c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

{'eval_loss': 0.3173596262931824, 'eval_accuracy': 0.549, 'eval_f1': 0.6639333333333333, 'eval_precision': 0.63475, 'eval_recall': 0.7375, 'eval_runtime': 166.5395, 'eval_samples_per_second': 6.005, 'eval_steps_per_second': 0.751, 'epoch': 4.0}
{'train_runtime': 2959.0582, 'train_samples_per_second': 1.352, 'train_steps_per_second': 0.169, 'train_loss': 0.3118371276855469, 'epoch': 4.0}


TrainOutput(global_step=500, training_loss=0.3118371276855469, metrics={'train_runtime': 2959.0582, 'train_samples_per_second': 1.352, 'train_steps_per_second': 0.169, 'total_flos': 213787324608000.0, 'train_loss': 0.3118371276855469, 'epoch': 4.0})

In [22]:
trainer.evaluate()

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.3241300582885742,
 'eval_accuracy': 0.702,
 'eval_f1': 0.702,
 'eval_precision': 0.702,
 'eval_recall': 0.702,
 'eval_runtime': 171.9938,
 'eval_samples_per_second': 5.814,
 'eval_steps_per_second': 0.727,
 'epoch': 4.0}

In [23]:
#test_metrics = trainer.evaluate(encoded_sentences_validation)
#print("Évaluation sur le jeu de test:", test_metrics)


In [24]:
#test_metrics = trainer.evaluate(encoded_sentences_validation)
#print("Évaluation sur le jeu de test:", test_metrics)


In [25]:
# Phrase à tester
text = "European citizens want to welcome migrants in need of international protection and want to welcome people who will contribute to our economy, but they are worried that we cannot manage migration, that we cannot control it,” added the commissioner."

# Tokeniser le texte
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Faire une prédiction
model.eval()  # Assurez-vous que le modèle est en mode évaluation
with torch.no_grad():
    outputs = model(**inputs)

# Appliquer la fonction sigmoid pour obtenir des probabilités (multi-label)
logits = outputs.logits
probabilities = torch.sigmoid(logits)

# Afficher les probabilités pour chaque classe
print(probabilities)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


tensor([[0.0636, 0.1046, 0.1658, 0.0695, 0.6131]])
